# Loan Shark Escape Demo\nThis notebook compares a naive debt-rollover policy with a strategic policy in `lse-medium`.

## Judge Narrative Target\n- Naive agent: spiral lock around month 11, high fee burn, TRAPPED.\n- Smart strategy: uses escape windows, exits early, much lower fee burn, ESCAPED.\n\nRun the cells below to produce real run numbers from your environment build.

In [ ]:
import asyncio\nimport json\nfrom typing import Any, Callable\n\nimport pandas as pd\n\nfrom client import LoanSharkEscapeEnv\n\nAPI_BASE_URL = "http://localhost:8000"\nTASK_ID = "lse-medium"

In [ ]:
async def run_episode(policy: Callable[[dict[str, Any]], int], label: str) -> dict[str, Any]:\n    async with LoanSharkEscapeEnv(API_BASE_URL) as env:\n        obs = await env.reset(TASK_ID)\n        done = False\n        steps = 0\n        reward_sum = 0.0\n\n        while not done and steps < 60:\n            action = int(policy(obs))\n            payload = await env.step({"action": action})\n            reward_sum += float(payload.get("reward", 0.0))\n            done = bool(payload.get("done", False))\n            obs = payload.get("observation", payload)\n            steps += 1\n\n        grade = await env.evaluate()\n        state = await env.state()\n\n    return {\n        "agent": label,\n        "months": steps,\n        "total_reward": round(reward_sum, 2),\n        "fees": round(float(state.get("total_fees_paid", 0.0)), 2),\n        "spiral_lock": bool(state.get("spiral_lock", False)),\n        "escaped": bool(state.get("all_loans_cleared", False)),\n        "grader_reward": grade.get("reward", 0.0),\n        "grader": grade,\n    }

In [ ]:
def naive_policy(_obs: dict[str, Any]) -> int:\n    return 1\n\ndef smart_scripted_policy(obs: dict[str, Any]) -> int:\n    months_remaining = int(obs.get("months_remaining", 0))\n    month = 18 - months_remaining\n    routes = obs.get("escape_routes", {})\n\n    if routes.get("credit_union_available", False):\n        return 2\n    if routes.get("ngo_help_available", False) and month in (6, 12):\n        return 3\n    if obs.get("cash_on_hand", 0.0) > 15000:\n        return 0\n    return 1

In [ ]:
naive = asyncio.run(run_episode(naive_policy, "Naive (always action=1)"))\nsmart = asyncio.run(run_episode(smart_scripted_policy, "Smart (escape-window strategy)"))\n\ntable = pd.DataFrame([naive, smart])[["agent", "months", "fees", "spiral_lock", "escaped", "grader_reward"]]\ndisplay(table)\n\nnaive_fees = float(naive["fees"])\nsmart_fees = float(smart["fees"])\nif naive_fees > 0:\n    reduction = ((naive_fees - smart_fees) / naive_fees) * 100\nelse:\n    reduction = 0.0\n\nprint(f"Fee reduction: {reduction:.2f}%")\nprint("\nNaive grader:\n", json.dumps(naive["grader"], indent=2))\nprint("\nSmart grader:\n", json.dumps(smart["grader"], indent=2))